In [1]:
from pathlib import Path

# 1. Definir la ruta de la carpeta 'logs_basicos' usando Pathlib
CARPETA_LOGS = Path.cwd() / "logs_basicos"

# 2. Crear la carpeta físicamente en tu Mac (exist_ok=True evita que falle si ya existe)
CARPETA_LOGS.mkdir(exist_ok=True)

# 3. Definir el nombre del archivo de texto adentro de esa carpeta
archivo_salida = CARPETA_LOGS / "hola_mundo.txt"

# 4. Escribir un texto simple adentro del archivo
with open(archivo_salida, "w", encoding="utf-8") as file:
    file.write("¡Automatización básica completada con éxito localmente!\n")

print(f"✅ Archivo creado con éxito en: {archivo_salida}")

✅ Archivo creado con éxito en: /Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/containers/second_container/third_version/next_exercise/example_2/logs_basicos/hola_mundo.txt


In [2]:
%%writefile app_basico.py
from pathlib import Path

# Buscamos la ruta interna del contenedor
CARPETA_LOGS = Path.cwd() / "logs_basicos"
CARPETA_LOGS.mkdir(exist_ok=True)

archivo_salida = CARPETA_LOGS / "hola_mundo.txt"

# Escribimos el archivo de texto
with open(archivo_salida, "w", encoding="utf-8") as file:
    file.write("¡Contenedor básico ejecutado con éxito!\n")

print(f"✅ Archivo creado dentro del contenedor en: {archivo_salida}")

Writing app_basico.py


In [3]:
!docker build -f Dockerfile_basico -t contenedor-basico .

[+] Building 0.0s (0/1)                                                         
[+] Building 0.1s (1/2)                                                         
 => [internal] load build definition from Dockerfile_basico                0.0s
 => => transferring dockerfile: 2B                                         0.0s
failed to solve with frontend dockerfile.v0: failed to read dockerfile: open /var/lib/docker/tmp/buildkit-mount1860461015/Dockerfile_basico: no such file or directory


In [4]:
%%writefile Dockerfile_basico
FROM python:3.11-slim
WORKDIR /app
COPY app_basico.py .
CMD ["python", "app_basico.py"]

Writing Dockerfile_basico


In [ ]:
!docker build -f Dockerfile_basico -t contenedor-basico .

In [6]:
%%writefile requirements.txt
boto3==1.34.0
moto==5.0.0

Writing requirements.txt


In [7]:
%%writefile app_cloud.py
import boto3
from moto import mock_aws
from pathlib import Path
from datetime import datetime

# 1. Definimos las rutas locales temporales con Pathlib
BASE_DIR = Path.cwd()
LOCAL_LOGS = BASE_DIR / "temp_logs"
LOCAL_LOGS.mkdir(exist_ok=True)

@mock_aws
def ejecutar_pipeline_cloud():
    print("☁️ [AWS] Iniciando entorno virtual simulado de S3...")
    
    # 2. Inicializar el cliente de AWS S3
    s3_client = boto3.client("s3", region_name="us-east-1")
    
    # 3. Crear el "Bucket" (la cubeta de almacenamiento en la nube)
    nombre_bucket = "reportes-flota-enterprise"
    s3_client.create_bucket(Bucket=nombre_bucket)
    print(f"📦 [S3] Bucket '{nombre_bucket}' creado con éxito en AWS Virtual.")
    
    # 4. Crear el reporte localmente primero usando Pathlib
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    archivo_local = LOCAL_LOGS / f"diagnostico_{timestamp}.txt"
    
    with open(archivo_local, "w", encoding="utf-8") as file:
        file.write(f"=== REPORTE SUBIDO A AWS S3 ===\n")
        file.write(f"Fecha de ejecución: {datetime.now()}\n")
        file.write(f"Estado de la Flota: Operativa / Sin Errores Críticos.\n")
        
    print(f"✍️ [LOCAL] Reporte temporal creado en: {archivo_local.name}")
    
    # 5. SUBIR EL ARCHIVO A LA NUBE (La prueba de fuego de Boto3)
    # Convertimos la ruta de Pathlib a string porque AWS requiere texto
    nombre_objeto_s3 = f"logs/año=2026/{archivo_local.name}"
    
    s3_client.upload_file(
        Filename=str(archivo_local),
        Bucket=nombre_bucket,
        Key=nombre_objeto_s3
    )
    print(f"🚀 [CLOUD] ¡Archivo subido exitosamente a S3 de AWS! Destino: s3://{nombre_bucket}/{nombre_objeto_s3}")

    # 6. VERIFICACIÓN: Listar los archivos que están en la nube para confirmar
    print("\n🔍 [VERIFICACIÓN] Consultando el Bucket de AWS S3...")
    respuesta = s3_client.list_objects_v2(Bucket=nombre_bucket)
    
    for objeto in respuesta.get("Contents", []):
        print(f"📌 Encontrado en la nube S3 -> Tamaño: {objeto['Size']} bytes | Llave: {objeto['Key']}")

if __name__ == "__main__":
    print("🚀 --- INICIANDO PIPELINE DE AUTOMATIZACIÓN CLOUD ---")
    ejecutar_pipeline_cloud()
    print("🏁 --- PIPELINE CLOUD FINALIZADO CON ÉXITO ---")

Writing app_cloud.py


In [8]:
%%writefile Dockerfile_cloud
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app_cloud.py .
CMD ["python", "app_cloud.py"]

Writing Dockerfile_cloud


In [ ]:
!docker build -f Dockerfile_cloud -t pipeline-cloud .


In [10]:
!docker run --rm pipeline-cloud

🚀 --- INICIANDO PIPELINE DE AUTOMATIZACIÓN CLOUD ---
☁️ [AWS] Iniciando entorno virtual simulado de S3...
📦 [S3] Bucket 'reportes-flota-enterprise' creado con éxito en AWS Virtual.
✍️ [LOCAL] Reporte temporal creado en: diagnostico_20260617_041947.txt
🚀 [CLOUD] ¡Archivo subido exitosamente a S3 de AWS! Destino: s3://reportes-flota-enterprise/logs/año=2026/diagnostico_20260617_041947.txt

🔍 [VERIFICACIÓN] Consultando el Bucket de AWS S3...
📌 Encontrado en la nube S3 -> Tamaño: 135 bytes | Llave: logs/año=2026/diagnostico_20260617_041947.txt
🏁 --- PIPELINE CLOUD FINALIZADO CON ÉXITO ---
